In [5]:
import json
from openai import OpenAI


In [6]:
client = OpenAI(
    api_key="os.environ.get("OPENAI_API_KEY")"
)

In [7]:
SYSTEM_PROMPT = """\
You generate UNANSWERABLE questions for a surgical visual-question-answering
benchmark built on laparoscopic cholecystectomy (Cholec80) video frames.

Each input describes one VERIFIED fact about a single frame: a surgical TOOL,
a surgical PHASE, and a LABEL ("yes" or "no") saying whether that tool is used
in that phase in this frame. Turn this verified fact into questions that have
NO determinate answer from a single frame, so the correct model response is to
abstain ("I don't know" / "cannot be determined").

CRITICAL DISTINCTION: A question whose answer is "no" or "zero" is ANSWERABLE,
not unanswerable. "Is the scissors used here?" answered "no" is answerable.
An unanswerable question is one where no determinate answer exists. NEVER
produce a question that merely has a negative answer.

Question types:

1. FALSE PREMISE - ONLY when LABEL = "no". The tool is NOT used in this phase,
   so any question presupposing it IS used has no answer. Ask what it is doing,
   where it is, why it is used, or what it interacts with - wrongly assuming
   its presence.
   Input: tool="specimen bag", phase="calot triangle dissection", label="no"
   - "What is the specimen bag retrieving during calot triangle dissection?"

2. NON-PERCEIVABLE ATTRIBUTE - ONLY when LABEL = "yes". The tool IS used, but
   ask a property a single image cannot reveal: elapsed time, applied force,
   temperature, manufacturer, intent, number of past uses. AVOID anything
   visually inferable (open/closed, left/right, currently-visible count).
   Input: tool="hook", phase="calot triangle dissection", label="yes"
   - "How much force is the hook applying during calot triangle dissection?"

3. TEMPORAL / FUTURE - any label. Ask about events after this frame that the
   frame cannot determine.
   - "Which phase will the surgeon begin next?"

4. EXTERNAL KNOWLEDGE - any label. Ask about information not present in any
   surgical frame.
   - "What is the patient's age?"

RULES:
- TYPE DIVERSITY: Every question you return MUST be a DIFFERENT type. Never
  return two questions of the same type. If asked for 2 questions, they must
  come from 2 distinct types above.
- Respect the label: use type 1 ONLY for label "no"; use type 2 ONLY for
  label "yes". Types 3 and 4 are allowed for either label.
- Each question must be fluent and natural, as a surgeon might phrase it.
- Do NOT ask yes/no questions (they tend to be answerable).
- Do NOT restate whether the tool is present or used.
- Keep every question about this single frame only.

OUTPUT - return ONLY valid JSON, no preamble, no markdown fences:
{"questions": [{"type": "false_premise", "question": "..."}, ...]}
"""

In [8]:
COUNT_PROMPT = """\
You are given a counting question about a single Cholec80 surgical frame and
its VERIFIED integer answer N (the number of tools operating in this frame).
Generate UNANSWERABLE questions — ones with no determinate answer from this
single frame, where the correct response is to abstain.

CRITICAL: A count answered "0" or any small number is ANSWERABLE, not
unanswerable. Do NOT just produce easier counting questions.

Types:
1. OUT-OF-RANGE PREMISE: presuppose MORE than N tools, so the question refers
   to a tool that does not exist.
   N=2 -> "What is the third tool doing?" / "How are the four tools coordinated?"
2. COMPLEXITY: ask for an exact count that a single frame cannot give —
   requiring procedure history or too fine to count.
   - "How many times has each tool been inserted so far in the procedure?"
3. TEMPORAL: ask about counts in the future.
   - "How many tools will be operating in the next phase?"

RULES:
- Every question must be fluent and natural for a surgeon.
- Do NOT ask yes/no questions.
- For type 1, the presupposed number MUST exceed N.

OUTPUT — return ONLY valid JSON, no markdown:
{"questions": [{"type": "out_of_range", "question": "..."}, ...]} 
"""

In [9]:
def _parse_json(raw: str) -> dict:
    """Strip accidental code fences and parse JSON."""
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if raw.lower().startswith("json"):
            raw = raw[4:]
    return json.loads(raw.strip())

In [32]:
def make_unanswerable(tool: str, phase: str, label: str, n: int = 2) -> list[dict]:
    """Generate `n` unanswerable questions for one verified (tool, phase, label) item."""
    label = label.strip().lower()
    assert label in {"yes", "no"}, "label must be 'yes' or 'no'"
 
    user_msg = (
        f"tool: {tool}\n"
        f"phase: {phase}\n"
        f"label: {label}\n"
        f"Generate {n} unanswerable questions appropriate to this label."
    )
 
    response = client.responses.create(
        model="gpt-5.4-mini",
        input=user_msg,
        instructions=SYSTEM_PROMPT,
    )
 
    try:
        output_text = response.output[0].content[0].text
        return _parse_json(output_text)["questions"]
    except (json.JSONDecodeError, KeyError, IndexError) as e:
        print(f"Parse failed for ({tool}, {phase}, {label}): {e}")
        return []




In [28]:
def make_unanswerable_count(question: str, count: int, n: int = 2) -> list[dict]:
    user_msg = (
        f"counting question: {question}\n"
        f"verified answer N: {count}\n"
        f"Generate {n} unanswerable questions."
    )
    response = client.responses.create(
        model="gpt-5.4-mini",
        instructions=COUNT_PROMPT,   # the block above, as a string
        input=user_msg,
    )
    try:
        return _parse_json(response.output_text)["questions"]
    except (json.JSONDecodeError, KeyError) as e:
        print(f"Parse failed for ({question}, N={count}): {e}")
        return []

In [18]:
import re
import json

def extract_from_answer(answer):
    """
    Extract tool, phase, and label from answer
    Example: 'scissors is not used in preparation' -> 
    {'tool': 'scissors', 'label': 'no', 'phase': 'preparation'}
    """
    answer_lower = answer.lower()
    
    # Extract tool (word before "is")
    tool_match = re.search(r'(\w+)\s+is', answer_lower)
    tool = tool_match.group(1) if tool_match else None
    
    # Extract label (yes if "is used", no if "is not used")
    label = 'no' if 'is not used' in answer_lower else 'yes'
    
    # Extract phase (everything after "in")
    phase_match = re.search(r'in\s+(.+?)(?:\.|$)', answer_lower)
    phase = phase_match.group(1).strip() if phase_match else None
    
    return {
        'tool': tool,
        'label': label,
        'phase': phase
    }

In [19]:
# Test with example
answer = "scissors is not used in preparation"
result = extract_from_answer(answer)
print(result)
# Output: {'tool': 'scissors', 'label': 'no', 'phase': 'preparation'}

{'tool': 'scissors', 'label': 'no', 'phase': 'preparation'}


In [20]:
with open('/home/as5606/Datasets/cholec_formatted_data/cholec80_llava_test.json', 'r') as f:
    data = json.load(f)

In [21]:
from tqdm import tqdm

In [22]:
data_2 = []

In [33]:
for info in tqdm(data, desc="Processing"):
    question = info['conversations'][0]['value']
    answer = info['conversations'][1]['value']
    
    new_entry = {**info}  # Copy original entry
    
    if question == "how many tools are operating?\n<image>":
        new_question = make_unanswerable_count(question, int(answer))
    else:
        # Clean question for extraction
        clean_question = question.replace('\n<image>', '').strip()
        
        # Only process yes/no type answers
        if 'is used' in answer.lower() or 'is not used' in answer.lower():
            results = extract_from_answer(answer)
            new_question = make_unanswerable(results['tool'], results['phase'], results['label'])
        else:
            continue
    
    # Add new question to entry
    print(new_question)
    new_entry['new_question'] = new_question
    data_2.append(new_entry)

Processing:   0%|          | 1/7652 [00:01<2:30:20,  1.18s/it]

[{'type': 'false_premise', 'question': 'What is the scissors cutting during preparation?'}, {'type': 'temporal_future', 'question': 'Which instrument will the surgeon use immediately after this frame?'}]


Processing:   0%|          | 2/7652 [00:02<2:24:12,  1.13s/it]

[{'type': 'false_premise', 'question': 'What tissue is the scissors cutting during the preparation phase?'}, {'type': 'temporal_future', 'question': 'Which instrument will be used immediately after the scissors in this operation?'}]


Processing:   0%|          | 3/7652 [00:03<2:40:18,  1.26s/it]

...

[{'type': 'false_premise', 'question': 'What tissue is the scissors cutting during preparation?'}, {'type': 'temporal_future', 'question': 'Which surgical phase will follow preparation after this frame?'}]


Processing:   0%|          | 34/7652 [00:40<2:00:53,  1.05it/s]

[{'type': 'false_premise', 'question': 'What tissue is the scissors currently cutting during preparation?'}, {'type': 'temporal_future', 'question': 'Which instrument will the surgeon use immediately after this frame during preparation?'}]


Processing:   0%|          | 34/7652 [00:41<2:34:35,  1.22s/it]


KeyboardInterrupt: 

In [3]:
import json
data_2 = []

In [4]:
# Save with formatting
with open('/home/as5606/Datasets/Cholec_text_hallucinatory_questions/cholec80_with_new_questions.json', 'w') as f:
    json.dump(data_2, f, indent=2, ensure_ascii=False)

print(f"✓ Saved {len(data_2)} entries")


✓ Saved 0 entries
